# Setup & Introduction 

In [1]:
# ============================================================================
# NOTEBOOK 4: STREAMLIT DEPLOYMENT
# ============================================================================
"""
Olist Marketplace Integrity Audit - Streamlit Web App

Purpose:
    Deploy the trained Logistic Regression model as an interactive web application
    for predicting first-time customer drop-off risk.

Deliverables:
    1. Interactive prediction interface
    2. Risk level classification (Low/Medium/High/Critical)
    3. Business recommendations
    4. Feature importance visualization
    
Deployment:
    - Local testing: streamlit run app.py
    - Production: Streamlit Cloud (streamlit.io)
    
Author: Reynold Takura Choruma
Date: March 2026
"""

import os
from pathlib import Path

print("="*70)
print("NOTEBOOK 4: STREAMLIT DEPLOYMENT")
print("="*70)
print("\n Creating web application for model deployment")
print(" Interactive prediction interface")
print(" Business recommendations engine")

NOTEBOOK 4: STREAMLIT DEPLOYMENT

 Creating web application for model deployment
 Interactive prediction interface
 Business recommendations engine


# Create Folder Structure 

In [2]:
# ============================================================================
# CREATE STREAMLIT APP FOLDER
# ============================================================================

import os

# Create streamlit_app folder if it doesn't exist
os.makedirs('../streamlit_app', exist_ok=True)

print(" Created folder: streamlit_app/")
print("\nFolder structure:")
print("Olist-Customer-Dropoff-Prediction/")
print("├── streamlit_app/        ← NEW!")
print("│   ├── app.py           (will be created)")
print("│   └── requirements.txt (will be created)")
print("├── outputs/models/")
print("│   └── logistic_regression_final.pkl")
print("└── notebooks/")

 Created folder: streamlit_app/

Folder structure:
Olist-Customer-Dropoff-Prediction/
├── streamlit_app/        ← NEW!
│   ├── app.py           (will be created)
│   └── requirements.txt (will be created)
├── outputs/models/
│   └── logistic_regression_final.pkl
└── notebooks/


In [3]:
%%writefile ../streamlit_app/app.py

import streamlit as st
import pandas as pd
import numpy as np
import pickle
import os

# Page configuration
st.set_page_config(
    page_title="Olist Retention Predictor",
    page_icon="🛒",
    layout="wide"
)

# Title
st.title("🛒 Olist Customer Retention Predictor")
st.markdown("### Predict first-time customer drop-off risk")

st.markdown("""
**Problem:** 95% of Olist first-time customers never make a second purchase.

**Solution:** This tool predicts drop-off risk based on first order characteristics.

**How it works:**
1. Enter customer's first order details
2. Get instant drop-off risk prediction
3. Receive retention recommendations
""")

st.divider()

# ============================================================================
# LOAD MODEL - FIXED PATH
# ============================================================================

@st.cache_resource
def load_model():
    """Load the trained Logistic Regression model"""
    # Get the absolute path - works from any directory
    current_dir = os.path.dirname(os.path.abspath(__file__))
    project_root = os.path.dirname(current_dir)
    model_path = os.path.join(project_root, 'outputs', 'models', 'logistic_regression_final.pkl')
    
    with open(model_path, 'rb') as f:
        model = pickle.load(f)
    return model

try:
    model = load_model()
    st.success("✅ Model loaded successfully!")
except Exception as e:
    st.error(f"❌ Error loading model: {e}")
    st.stop()

# ============================================================================
# INPUT FORM
# ============================================================================

st.header("📝 Enter Customer First Order Details")

col1, col2, col3 = st.columns(3)

with col1:
    st.subheader("🚚 Delivery")
    delivery_delay = st.number_input("Delivery Delay (days)", -30, 100, 0)
    total_days = st.number_input("Total Days to Delivery", 0, 100, 10)

with col2:
    st.subheader("💰 Economics")
    freight_pct = st.slider("Freight % of Order", 0.0, 80.0, 15.0)
    num_items = st.number_input("Number of Items", 1, 20, 1)
    price_per_item = st.number_input("Avg Price per Item (R$)", 1.0, 10000.0, 100.0)
    uses_installments = st.checkbox("Uses Installments")

with col3:
    st.subheader("👤 Customer & Product")
    is_southeast = st.checkbox("Southeast Brazil", value=True)
    is_repeatable = st.checkbox("Repeatable Category")
    is_heavy = st.checkbox("Heavy Product (>5kg)")
    has_comment = st.checkbox("Left Review Comment")
    is_holiday = st.checkbox("Holiday Season")
    is_weekend = st.checkbox("Weekend Purchase")

st.divider()

# ============================================================================
# PREDICT BUTTON
# ============================================================================

if st.button("🔮 Predict Drop-off Risk", type="primary"):
    
    # Create feature vector (simplified for now - matches your 33 features)
    # NOTE: You'll need to add ALL 33 features to match training exactly
    features = pd.DataFrame({
        'delivery_delay': [delivery_delay],
        'is_late_delivery': [int(delivery_delay > 0)],
        'is_very_late': [int(delivery_delay > 10)],
        'is_early_delivery': [int(delivery_delay < 0)],
        'freight_pct': [freight_pct],
        'is_high_freight': [int(freight_pct > 20)],
        'num_items': [num_items],
        'price_per_item': [price_per_item],
        'uses_installments': [int(uses_installments)],
        'is_southeast': [int(is_southeast)],
        'is_repeatable_category': [int(is_repeatable)],
        'is_heavy_product': [int(is_heavy)],
        'has_comment': [int(has_comment)],
        'purchase_month': [11],  # Default
        'purchase_day_of_week': [0],  # Default
        'is_weekend': [int(is_weekend)],
        'is_holiday_season': [int(is_holiday)],
        'days_to_delivery': [total_days],
        'cluster_0': [0],  # Default
        'cluster_1': [0],  # Default
        'payment_boleto': [0],  # Default
        'payment_credit_card': [1],  # Default
        'payment_debit_card': [0],  # Default
        'state_BA': [0],
        'state_DF': [0],
        'state_ES': [0],
        'state_GO': [0],
        'state_MG': [0],
        'state_PR': [0],
        'state_RJ': [0],
        'state_RS': [0],
        'state_SC': [0],
        'state_SP': [int(is_southeast)]
    })
    
    try:
        # Make prediction
        prediction_proba = model.predict_proba(features)[0]
        drop_off_prob = prediction_proba[1] * 100  # Convert to percentage
        
        # Display result
        st.subheader("📊 Prediction Results")
        
        # Risk gauge
        col1, col2 = st.columns([1, 2])
        
        with col1:
            st.metric("Drop-off Probability", f"{drop_off_prob:.1f}%")
            
            if drop_off_prob >= 98:
                st.error("🔴 CRITICAL RISK")
            elif drop_off_prob >= 95:
                st.warning("🟠 HIGH RISK")
            elif drop_off_prob >= 90:
                st.info("🟡 MEDIUM RISK")
            else:
                st.success("🟢 LOW RISK")
        
        with col2:
            st.markdown("### 💡 Recommendations")
            
            if freight_pct > 20:
                st.markdown("- ⚠️ **High freight cost detected** - Consider offering free shipping")
            
            if not is_repeatable:
                st.markdown("- ⚠️ **Non-repeatable category** - Focus on cross-sell opportunities")
            
            if not uses_installments:
                st.markdown("- 💡 **Promote installment payments** - Increases retention by 26%")
            
            if is_holiday:
                st.markdown("- ✅ **Holiday purchase** - These customers are 61% more likely to return!")
            
            st.markdown("- 📧 **Send follow-up email within 7 days**")
            st.markdown("- 🎁 **Offer discount on next purchase**")
        
        st.balloons()
        
    except Exception as e:
        st.error(f"❌ Prediction error: {e}")

# Footer
st.divider()
st.markdown("""
<div style='text-align: center; color: gray; padding: 20px;'>
    <p>Olist Marketplace Integrity Audit | Developed by Reynold Choruma | March 2026</p>
    <p>Model: Logistic Regression | PR AUC: 0.9654 | Minority Recall: 62.6%</p>
</div>
""", unsafe_allow_html=True)


Overwriting ../streamlit_app/app.py


# Create Requirements TXT

In [4]:
%%writefile ../streamlit_app/requirements.txt

streamlit==1.31.0
pandas==2.1.4
numpy==1.26.3
scikit-learn==1.4.0
plotly==5.18.0

Overwriting ../streamlit_app/requirements.txt


In [5]:
# ============================================================================
# STREAMLIT APP CREATED - TESTING INSTRUCTIONS
# ============================================================================

print("="*70)
print("✅ STREAMLIT APP SUCCESSFULLY CREATED!")
print("="*70)

print("\n📁 Files Created:")
print("   ✅ streamlit_app/app.py")
print("   ✅ streamlit_app/requirements.txt")

print("\n🚀 TO TEST YOUR APP LOCALLY:")
print("="*70)
print("\nOption 1: From Terminal/Command Line")
print("-" * 70)
print("   cd streamlit_app")
print("   streamlit run app.py")

print("\nOption 2: From Jupyter (run cell below)")
print("-" * 70)
print("   Uncomment and run Cell 6")

print("\n📱 The app will open at: http://localhost:8501")

print("\n💡 To stop the app: Press Ctrl+C in terminal")

print("\n" + "="*70)
print("READY TO TEST!")
print("="*70)

✅ STREAMLIT APP SUCCESSFULLY CREATED!

📁 Files Created:
   ✅ streamlit_app/app.py
   ✅ streamlit_app/requirements.txt

🚀 TO TEST YOUR APP LOCALLY:

Option 1: From Terminal/Command Line
----------------------------------------------------------------------
   cd streamlit_app
   streamlit run app.py

Option 2: From Jupyter (run cell below)
----------------------------------------------------------------------
   Uncomment and run Cell 6

📱 The app will open at: http://localhost:8501

💡 To stop the app: Press Ctrl+C in terminal

READY TO TEST!


# Deployment Instruction

In [6]:
# ============================================================================
# DEPLOYMENT TO STREAMLIT CLOUD
# ============================================================================

print("="*70)
print("DEPLOYMENT INSTRUCTIONS")
print("="*70)

print("\n📦 STEP 1: PUSH TO GITHUB")
print("-" * 70)
print("cd ~/Desktop/Olist-Customer-Dropoff-Prediction/")
print("git add .")
print('git commit -m "Add Streamlit deployment app"')
print("git push origin main")

print("\n🌐 STEP 2: DEPLOY TO STREAMLIT CLOUD")
print("-" * 70)
print("1. Go to: https://streamlit.io/cloud")
print("2. Sign in with GitHub")
print("3. Click 'New app'")
print("4. Select your repository: ReynoldT92/Olist-Customer-Drop-Off-Prediction")
print("5. Main file path: streamlit_app/app.py")
print("6. Click 'Deploy'")

print("\n✨ STEP 3: SHARE YOUR APP")
print("-" * 70)
print("You'll get a URL like:")
print("https://olist-retention-predictor.streamlit.app")
print("\nAdd this to:")
print("• Your GitHub README")
print("• Your resume/portfolio")
print("• Your presentation for Sabina")

print("\n" + "="*70)
print("✅ NOTEBOOK 4 COMPLETE - APP READY FOR DEPLOYMENT!")
print("="*70)

DEPLOYMENT INSTRUCTIONS

📦 STEP 1: PUSH TO GITHUB
----------------------------------------------------------------------
cd ~/Desktop/Olist-Customer-Dropoff-Prediction/
git add .
git commit -m "Add Streamlit deployment app"
git push origin main

🌐 STEP 2: DEPLOY TO STREAMLIT CLOUD
----------------------------------------------------------------------
1. Go to: https://streamlit.io/cloud
2. Sign in with GitHub
3. Click 'New app'
4. Select your repository: ReynoldT92/Olist-Customer-Drop-Off-Prediction
5. Main file path: streamlit_app/app.py
6. Click 'Deploy'

✨ STEP 3: SHARE YOUR APP
----------------------------------------------------------------------
You'll get a URL like:
https://olist-retention-predictor.streamlit.app

Add this to:
• Your GitHub README
• Your resume/portfolio
• Your presentation for Sabina

✅ NOTEBOOK 4 COMPLETE - APP READY FOR DEPLOYMENT!


In [7]:
# Check what's on line 354
with open('../streamlit_app/app.py', 'r') as f:
    lines = f.readlines()
    
print("Lines 350-360:")
for i in range(349, min(360, len(lines))):
    print(f"{i+1}: {lines[i]}", end='')


Lines 350-360:


In [8]:
%%writefile ../streamlit_app/app.py

import streamlit as st
import pandas as pd
import pickle
from pathlib import Path

# Page config
st.set_page_config(
    page_title="Olist Retention Predictor",
    page_icon="🛒",
    layout="wide"
)

# Title
st.title("🛒 Olist Customer Retention Predictor")
st.markdown("### Predict first-time customer drop-off risk")

# Load model
@st.cache_resource
def load_model():
    model_path = Path(__file__).parent.parent / 'outputs' / 'models' / 'logistic_regression_final.pkl'
    with open(model_path, 'rb') as f:
        return pickle.load(f)

try:
    model = load_model()
    st.success("✅ Model loaded successfully!")
except Exception as e:
    st.error(f"❌ Error loading model: {e}")
    st.stop()

st.divider()

# Simple test form
st.header("📝 Test Prediction")

col1, col2 = st.columns(2)

with col1:
    delivery_delay = st.number_input("Delivery Delay (days)", value=0)
    freight_pct = st.slider("Freight %", 0.0, 50.0, 15.0)
    num_items = st.number_input("Number of Items", 1, 10, 1)

with col2:
    is_repeatable = st.checkbox("Repeatable Category")
    is_holiday = st.checkbox("Holiday Season")
    uses_installments = st.checkbox("Uses Installments")

if st.button("🔮 Predict", type="primary"):
    st.success("Prediction functionality coming soon!")
    st.balloons()


Overwriting ../streamlit_app/app.py


In [9]:
# Fix the model path in app.py
with open('../streamlit_app/app.py', 'r') as f:
    content = f.read()

# Find and replace the model path line
old_path = "model_path = Path(__file__).parent.parent / 'outputs' / 'models' / 'logistic_regression_final.pkl'"
new_path = "model_path = Path(__file__).parent.parent / 'outputs' / 'models' / 'logistic_regression_final.pkl'"

# Actually, let's use an absolute path to be safe
import os
absolute_model_path = os.path.abspath('../outputs/models/logistic_regression_final.pkl')

# Replace the load_model function
old_function = """@st.cache_resource
def load_model():
    model_path = Path(__file__).parent.parent / 'outputs' / 'models' / 'logistic_regression_final.pkl'
    with open(model_path, 'rb') as f:
        return pickle.load(f)"""

new_function = f"""@st.cache_resource
def load_model():
    model_path = r'{absolute_model_path}'
    with open(model_path, 'rb') as f:
        return pickle.load(f)"""

content = content.replace(old_function, new_function)

# Write back
with open('../streamlit_app/app.py', 'w') as f:
    f.write(content)

print("✅ Fixed model path!")
print(f"✅ Model will be loaded from: {absolute_model_path}")


✅ Fixed model path!
✅ Model will be loaded from: /Users/reynoldtakurachoruma/Desktop/Olist-Customer-Dropoff-Prediction/outputs/models/logistic_regression_final.pkl


In [10]:
import os

model_path = '../outputs/models/logistic_regression_final.pkl'

if os.path.exists(model_path):
    print(f"✅ Model found at: {os.path.abspath(model_path)}")
else:
    print(f"❌ Model NOT found at: {os.path.abspath(model_path)}")
    print("\n📂 Checking what files exist in outputs/models/:")
    if os.path.exists('../outputs/models/'):
        files = os.listdir('../outputs/models/')
        for f in files:
            print(f"   • {f}")
    else:
        print("   ❌ outputs/models/ directory doesn't exist!")


✅ Model found at: /Users/reynoldtakurachoruma/Desktop/Olist-Customer-Dropoff-Prediction/outputs/models/logistic_regression_final.pkl


In [11]:
%%writefile ../streamlit_app/app.py

import streamlit as st
import pandas as pd
import numpy as np
import pickle
import os

# Page configuration
st.set_page_config(
    page_title="Olist Retention Predictor",
    page_icon="🛒",
    layout="wide"
)

# Custom CSS for styling
st.markdown("""
    <style>
    .main {padding: 2rem;}
    .stButton>button {
        width: 100%;
        background-color: #2ecc71;
        color: white;
        font-weight: bold;
        padding: 0.5rem;
        border-radius: 5px;
    }
    .stButton>button:hover {background-color: #27ae60;}
    </style>
    """, unsafe_allow_html=True)

# ============================================================================
# LOAD MODEL
# ============================================================================

@st.cache_resource
def load_model():
    """Load the trained Logistic Regression model"""
    current_dir = os.path.dirname(os.path.abspath(__file__))
    project_root = os.path.dirname(current_dir)
    model_path = os.path.join(project_root, 'outputs', 'models', 'logistic_regression_final.pkl')
    
    with open(model_path, 'rb') as f:
        model = pickle.load(f)
    return model

try:
    model = load_model()
    st.success("✅ Model loaded successfully!")
except Exception as e:
    st.error(f"❌ Error loading model: {e}")
    st.stop()

# ============================================================================
# HEADER
# ============================================================================

st.title("🛒 Olist Customer Retention Predictor")
st.markdown("### Predict first-time customer drop-off risk")

st.markdown("""
**Problem:** 95% of Olist first-time customers never make a second purchase.

**Solution:** This tool predicts which customers are at risk of dropping off 
based on their first order characteristics, enabling targeted retention interventions.

**How it works:**
1. Enter customer's first order details below
2. Click "Predict Drop-off Risk"
3. Get instant risk assessment and recommendations
""")

st.divider()

# ============================================================================
# INPUT FORM
# ============================================================================

st.header("📝 Enter Customer First Order Details")

col1, col2, col3 = st.columns(3)

with col1:
    st.subheader("🚚 Delivery Info")
    delivery_delay = st.number_input(
        "Delivery Delay (days)",
        min_value=-30,
        max_value=100,
        value=0,
        help="Negative = early, Positive = late"
    )
    
    days_to_delivery = st.number_input(
        "Total Days to Delivery",
        min_value=0,
        max_value=100,
        value=10
    )

with col2:
    st.subheader("💰 Order Economics")
    freight_pct = st.slider(
        "Freight % of Order Value",
        min_value=0.0,
        max_value=80.0,
        value=15.0,
        help="Shipping cost as % of order value"
    )
    
    num_items = st.number_input(
        "Number of Items",
        min_value=1,
        max_value=20,
        value=1
    )
    
    price_per_item = st.number_input(
        "Price per Item (R$)",
        min_value=1.0,
        max_value=10000.0,
        value=100.0
    )
    
    uses_installments = st.checkbox("Uses Installment Payment", value=False)

with col3:
    st.subheader("👤 Customer & Product")
    is_southeast = st.checkbox(
        "Southeast Brazil Customer",
        value=True,
        help="SP, RJ, MG, ES states"
    )
    
    is_repeatable_category = st.checkbox(
        "Repeatable Category",
        value=False,
        help="Health/beauty, books, pet supplies"
    )
    
    is_heavy_product = st.checkbox("Heavy Product (>5kg)", value=False)
    
    has_comment = st.checkbox("Left Review Comment", value=False)
    
    is_holiday_season = st.checkbox(
        "Holiday Season Purchase",
        value=False,
        help="November or December"
    )
    
    is_weekend = st.checkbox("Weekend Purchase", value=False)

st.divider()

# ============================================================================
# ADVANCED OPTIONS (COLLAPSED)
# ============================================================================

with st.expander("⚙️ Advanced Options (Optional)"):
    adv_col1, adv_col2 = st.columns(2)
    
    with adv_col1:
        purchase_month = st.selectbox(
            "Purchase Month",
            options=list(range(1, 13)),
            index=10  # November default
        )
        
        payment_type = st.radio(
            "Payment Type",
            options=["Credit Card", "Boleto", "Debit Card"],
            index=0
        )
    
    with adv_col2:
        cluster = st.selectbox(
            "Customer Segment",
            options=["Unknown", "Budget Shoppers", "High Freight/Risk"],
            index=0
        )
        
        state = st.selectbox(
            "Customer State",
            options=["SP", "RJ", "MG", "RS", "PR", "SC", "BA", "DF", "ES", "GO", "Other"],
            index=0
        )

# ============================================================================
# PREDICTION
# ============================================================================

if st.button(" Predict Drop-off Risk", type="primary"):
    
    # Calculate derived features
    is_late_delivery = int(delivery_delay > 0)
    is_very_late = int(delivery_delay > 10)
    is_early_delivery = int(delivery_delay < 0)
    is_high_freight = int(freight_pct > 20)
    
    # Payment type one-hot encoding
    payment_boleto = int(payment_type == "Boleto")
    payment_credit_card = int(payment_type == "Credit Card")
    payment_debit_card = int(payment_type == "Debit Card")
    
    # Cluster encoding
    cluster_0 = int(cluster == "Budget Shoppers")
    cluster_1 = int(cluster == "High Freight/Risk")
    
    # State encoding
    state_map = {
        'BA': 'state_BA', 'DF': 'state_DF', 'ES': 'state_ES',
        'GO': 'state_GO', 'MG': 'state_MG', 'PR': 'state_PR',
        'RJ': 'state_RJ', 'RS': 'state_RS', 'SC': 'state_SC',
        'SP': 'state_SP'
    }
    
    state_features = {
        'state_BA': 0, 'state_DF': 0, 'state_ES': 0, 'state_GO': 0,
        'state_MG': 0, 'state_PR': 0, 'state_RJ': 0, 'state_RS': 0,
        'state_SC': 0, 'state_SP': 0
    }
    
    if state in state_map:
        state_features[state_map[state]] = 1
    
    # Day of week calculation
    purchase_day_of_week = 5 if is_weekend else 2  # Saturday or Tuesday
    
    # Create feature vector matching training data (33 features)
    features = pd.DataFrame({
        'delivery_delay': [delivery_delay],
        'is_late_delivery': [is_late_delivery],
        'is_very_late': [is_very_late],
        'is_early_delivery': [is_early_delivery],
        'freight_pct': [freight_pct],
        'is_high_freight': [is_high_freight],
        'num_items': [num_items],
        'price_per_item': [price_per_item],
        'uses_installments': [int(uses_installments)],
        'is_southeast': [int(is_southeast)],
        'is_repeatable_category': [int(is_repeatable_category)],
        'is_heavy_product': [int(is_heavy_product)],
        'has_comment': [int(has_comment)],
        'purchase_month': [purchase_month],
        'purchase_day_of_week': [purchase_day_of_week],
        'is_weekend': [int(is_weekend)],
        'is_holiday_season': [int(is_holiday_season)],
        'days_to_delivery': [days_to_delivery],
        'cluster_0': [cluster_0],
        'cluster_1': [cluster_1],
        'payment_boleto': [payment_boleto],
        'payment_credit_card': [payment_credit_card],
        'payment_debit_card': [payment_debit_card],
        'state_BA': [state_features['state_BA']],
        'state_DF': [state_features['state_DF']],
        'state_ES': [state_features['state_ES']],
        'state_GO': [state_features['state_GO']],
        'state_MG': [state_features['state_MG']],
        'state_PR': [state_features['state_PR']],
        'state_RJ': [state_features['state_RJ']],
        'state_RS': [state_features['state_RS']],
        'state_SC': [state_features['state_SC']],
        'state_SP': [state_features['state_SP']]
    })
    
    try:
        # Make prediction
        prediction_proba = model.predict_proba(features)[0]
        drop_off_prob = prediction_proba[1] * 100  # Convert to percentage
        retention_prob = prediction_proba[0] * 100
        
        # Determine risk level
        if drop_off_prob >= 98:
            risk_level = "🔴 CRITICAL RISK"
            risk_color = "red"
        elif drop_off_prob >= 95:
            risk_level = "🟠 HIGH RISK"
            risk_color = "orange"
        elif drop_off_prob >= 90:
            risk_level = "🟡 MEDIUM RISK"
            risk_color = "blue"
        else:
            risk_level = "🟢 LOW RISK"
            risk_color = "green"
        
        # Display results
        st.divider()
        st.header("Prediction Results")
        
        # Metrics row
        metric_col1, metric_col2, metric_col3 = st.columns(3)
        
        with metric_col1:
            st.metric(
                "Drop-off Probability",
                f"{drop_off_prob:.1f}%",
                delta=f"{drop_off_prob - 95:.1f}% vs baseline",
                delta_color="inverse"
            )
        
        with metric_col2:
            st.metric(
                "Retention Probability",
                f"{retention_prob:.1f}%",
                delta=f"{retention_prob - 5:.1f}% vs baseline"
            )
        
        with metric_col3:
            if risk_color == "red":
                st.error(risk_level)
            elif risk_color == "orange":
                st.warning(risk_level)
            elif risk_color == "blue":
                st.info(risk_level)
            else:
                st.success(risk_level)
        
        # Recommendations
        st.subheader("💡 Personalized Recommendations")
        
        recommendations = []
        
        # Risk-specific recommendations
        if drop_off_prob >= 95:
            recommendations.append("🚨 **URGENT:** High-risk customer - activate retention protocol immediately")
            recommendations.append("📧 Send personalized follow-up email within 24 hours")
            recommendations.append("🎁 Offer 20% discount coupon valid for 7 days")
        
        # Feature-specific recommendations
        if freight_pct > 20:
            recommendations.append("📦 **High shipping cost detected** - Consider free shipping offer or subsidy")
        
        if not is_repeatable_category:
            recommendations.append("🔄 **Non-repeatable product** - Focus on cross-sell to recurring categories")
        
        if not uses_installments:
            recommendations.append("💳 **Promote installment payments** - Historical data shows 26% retention increase")
        
        if is_holiday_season:
            recommendations.append("✅ **Holiday purchase advantage** - These customers are 61% more likely to return!")
            recommendations.append("🎄 Send targeted holiday follow-up campaign")
        else:
            recommendations.append("📅 Consider seasonal promotion to re-engage")
        
        if is_late_delivery:
            recommendations.append("⏰ **Late delivery detected** - Issue apology credit or compensation")
        
        if not is_southeast:
            recommendations.append("🗺️ **Non-Southeast customer** - Extra attention needed for retention")
        
        # Display recommendations
        for rec in recommendations:
            st.markdown(f"- {rec}")
        
        # ROI Calculator
        st.subheader("💰 Retention ROI Estimate")
        
        roi_col1, roi_col2 = st.columns(2)
        
        with roi_col1:
            st.markdown("**Intervention Cost:** R$ 15 per customer")
            st.markdown("**Expected Success Rate:** 30%")
            st.markdown("**Customer LTV (if retained):** R$ 160")
        
        with roi_col2:
            expected_value = (retention_prob / 100) * 160 - 15
            roi = ((expected_value + 15) / 15 - 1) * 100 if expected_value > -15 else -100
            
            st.metric("Expected Value", f"R$ {expected_value:.2f}")
            st.metric("ROI", f"{roi:.1f}%")
            
            if expected_value > 0:
                st.success("✅ Intervention recommended - Positive ROI expected")
            else:
                st.warning("⚠️ Intervention not cost-effective at current probability")
        
        # Success animation
        st.balloons()
        
    except Exception as e:
        st.error(f"❌ Prediction error: {e}")
        st.error("Please check that all inputs are valid and try again.")

# ============================================================================
# FOOTER
# ============================================================================

st.divider()
st.markdown("""
<div style='text-align: center; color: gray; padding: 20px;'>
    <p><strong>Olist Marketplace Integrity Audit</strong> | Developed by Reynold Choruma | March 2026</p>
    <p>Model: Logistic Regression | PR AUC: 0.9654 | Minority Class Recall: 62.6%</p>
</div>
""", unsafe_allow_html=True)


Overwriting ../streamlit_app/app.py


In [12]:
# Remove the balloons from app.py
with open('../streamlit_app/app.py', 'r') as f:
    content = f.read()

# Remove the balloons line
content = content.replace("        # Success animation\n        st.balloons()", "        # Success - prediction complete")

# Write back
with open('../streamlit_app/app.py', 'w') as f:
    f.write(content)

print("✅ Balloons removed!")
print("✅ Now more professional for Sabina! 😂")


✅ Balloons removed!
✅ Now more professional for Sabina! 😂
